<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>0. Prepare Before Prod (normally already done)</h1>
</div>

<div style="background-color: #013220; color: white; padding: 10px;">
    <h2>0.0 Preprocessing Features</h2>
</div>

In [1]:
import pandas as pd
from py_files import PARAMS_LOADING, PREPROCESSING, PREDICTION
%load_ext autoreload
%autoreload 2

Excel_launcher_path = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\Launcher ML - 1M.xlsm"
params_principal, params_preprocessing, params_strat_selected, params_hyper_parameters = PARAMS_LOADING.get_all_params_from_excel(Excel_launcher_path)

In [2]:
Preprocessing_needed = False
if Preprocessing_needed:
    # Create X's variations (change features, sector dummies) and Y's forard returns as target to predict
    # Generate pickl "screen_ML_prod.pkl" as input for model's prediction in following steps
    PREPROCESSING.preprocess_data(params_principal, params_preprocessing)

<div style="background-color: #013220; color: white; padding: 10px;">
    <h2>0.1 Backtest => for getting all the history of ML Score</h2>
</div>

In [ ]:
# Step 1: Initialize parameters
input_transformed = pd.read_pickle(r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\input_files\screen_ML_prod.pkl")
params = PARAMS_LOADING.unpack_strategy_parameters(params_strat_selected)

# Step 2: Prepare initial dataset, including create label columns, fill nan values and filter rows only in universe
screen_label = PREDICTION.labellize_data_and_fill_nan(input_transformed, params)

# Step 3: Prediction and Generate Score ML for whole historical data (from 2010)
output_for_all_historic = PREDICTION.create_historic_Score_ML(screen_label,
                                                            params,
                                                            params_hyper_parameters,
                                                            update_score_ML=True,
                                                            output_file = "SCORE_ML_EUROPE", # This will also save excel and pickle file locally "SCORE_ML_2010_to_Today"
                                                            allow_multiprocessing=True)

<div style="background-color: #013220; color: white; padding: 10px;">
    <h2>0.2 Adding historical Score ML in screen agg</h2>
</div>

In [ ]:
#If we want to replace all the ML Score from the screen_aggregate with a new one, set True
rewrite_all_historical_ml_score = False
if rewrite_all_historical_ml_score:
    histo = pd.read_pickle(r'\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\SCORE_ML_2010_to_Today.pkl')[['Company SEDOL', 'Date','rank', 'Score ML']].dropna()
    histo.set_index(['Company SEDOL','Date'],inplace=True) #Place ces deux colonnes en index de histo

    screen_agg = pd.read_pickle(screen_path) #Lis screen
    screen_agg.reset_index(inplace=True) #Sort l'index
    screen_agg.set_index(['Company SEDOL','Date'],inplace=True) #Regle l'index avec le double index comme histo
    screen_agg.loc[histo.index, 'Score ML'] = histo['Score ML'] #Remplace la colonne de screen par celle de histo

    screen_agg.reset_index(inplace=True) #Sort l'index
    screen_agg.set_index('ISIN', inplace=True) #Regle l'index sur la colonne "ISIN"

    screen_agg.to_pickle(screen_path) #Export en pickle